# Cell 1: Core Imports & Augmentation Transforms

In [ ]:
import sys
import os
import random
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
from torchvision import transforms
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, roc_curve, auc, confusion_matrix, accuracy_score
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Core libraries initialized on {device} | Random seed locked to 42")

# Cell 2: Model Architecture (TSA-Net)

In [ ]:
class TSANet(nn.Module):
    def __init__(self, d_model=256):
        super(TSANet, self).__init__()
        
        efficientnet = models.efficientnet_b0(weights=None)
        self.v_backbone = efficientnet.features
        self.v_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.v_fc = nn.Linear(1280, d_model)
        
        self.a_backbone = nn.Sequential(
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Conv1d(128, d_model, kernel_size=3, padding=1),
            nn.BatchNorm1d(d_model),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(128)
        )
        
        class CrossAttention(nn.Module):
            def __init__(self, d_model=256, nhead=8):
                super().__init__()
                self.v_to_a_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
                self.a_to_v_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
                self.norm_v = nn.LayerNorm(d_model)
                self.norm_a = nn.LayerNorm(d_model)

            def forward(self, v_feat, a_feat):
                v_attn, _ = self.v_to_a_attn(query=v_feat, key=a_feat, value=a_feat)
                v_out = self.norm_v(v_feat + v_attn)
                
                a_attn, _ = self.a_to_v_attn(query=a_feat, key=v_feat, value=v_feat)
                a_out = self.norm_a(a_feat + a_attn)
                
                return v_out, a_out

        self.cross_attn = CrossAttention(d_model=d_model, nhead=8)
        
        self.classifier = nn.Sequential(
            nn.Linear(d_model * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, visual_inputs, audio_inputs):
        batch_size, num_frames, c, h, w = visual_inputs.shape
        v_x = visual_inputs.view(batch_size * num_frames, c, h, w)
        v_x = self.v_backbone(v_x)
        v_x = self.v_pool(v_x).flatten(1)
        v_x = self.v_fc(v_x)
        
        v_x = F.layer_norm(v_x, (v_x.size(-1),))
        v_feat = v_x.view(batch_size, num_frames, -1)
        
        a_x = self.a_backbone(audio_inputs)
        a_feat = a_x.transpose(1, 2)
        a_feat = F.layer_norm(a_feat, (a_feat.size(-1),))
        
        v_attn, a_attn = self.cross_attn(v_feat, a_feat)
        
        v_pooled = torch.mean(v_attn, dim=1)
        a_pooled = torch.mean(a_attn, dim=1)
        
        fused = torch.cat([v_pooled, a_pooled], dim=-1)
        logits = self.classifier(fused)
        
        return logits

model = TSANet(d_model=256).to(device)
print("✓ TSANet initialized successfully!")

# Cell 3: Weight Loading

In [ ]:
# Cell 3: Dynamic Checkpoint Finder & Loader
# NOTE: now prefers tsa_net_augmented_best.pth (the final, fully-trained
# augmented model your app.py actually deploys) over the earlier
# pre-augmentation checkpoints -- so if you're only here to train the
# audio-only head (skipping Cell 9's retraining loop), this loads the
# correct, already-trained weights directly.
def find_checkpoint():
    search_root = "/kaggle/input" if os.path.exists("/kaggle/input") else "."
    preferred_order = [
        "tsa_net_augmented_best.pth",     # final augmented model (preferred)
        "tsa_net_fakeavceleb_phase2.pth",
        "tsa_net_fakeavceleb_best.pth",
    ]
    found = {}
    for root, _, files in os.walk(search_root):
        for f in files:
            if f in preferred_order and f not in found:
                found[f] = os.path.join(root, f)
    for name in preferred_order:
        if name in found:
            return found[name], name
    return None, None

CHECKPOINT_PATH, CHECKPOINT_NAME = find_checkpoint()

if CHECKPOINT_PATH and os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    state_dict = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint else checkpoint
    model.load_state_dict(state_dict, strict=True)
    print(f"\u2713 Successfully loaded pre-trained weights from: {CHECKPOINT_PATH}")
    if CHECKPOINT_NAME == "tsa_net_augmented_best.pth":
        print("\u2713 This is the final augmented checkpoint -- Cell 9 (retraining) can be SKIPPED")
        print("  if you only need this notebook to train the audio-only head below.")
    else:
        print("\u26a0\ufe0f This is a pre-augmentation checkpoint -- Cell 9 still needs to run")
        print("  to produce the final augmented weights before training the audio-only head.")
else:
    print("\u274c ERROR: Checkpoint file not found under /kaggle/input!")
    print("Please attach your previous notebook output (e.g., MPFour or MPFive) in the right-hand 'Data' panel.")


# Cell 4: Augmented Dataset Pipeline

In [ ]:
def find_fakeavceleb_paths():
    search_root = "/kaggle/input" if os.path.exists("/kaggle/input") else "."
    for root, _, files in os.walk(search_root):
        if "meta_data.csv" in files:
            return os.path.join(root, "meta_data.csv"), root
    return None, None

METADATA_CSV_PATH, KAGGLE_DATASET_ROOT = find_fakeavceleb_paths()

class AugmentedFakeAVCelebDataset(Dataset):
    def __init__(self, csv_path, dataset_root, split='train', num_frames=5, augment=True):
        self.dataset_root = dataset_root
        self.num_frames = num_frames
        self.augment = augment
        
        df = pd.read_csv(csv_path)
        file_index = {}
        video_exts = ('.mp4', '.avi', '.mov', '.mkv')
        for root, _, files in os.walk(self.dataset_root):
            for f in files:
                if f.lower().endswith(video_exts):
                    file_index[f.lower()] = os.path.join(root, f)
        
        def resolve_file(rel_path):
            if pd.isna(rel_path): return None
            basename = os.path.basename(str(rel_path)).strip().lower()
            return file_index.get(basename, None)

        df['resolved_path'] = df['path'].apply(resolve_file)
        valid_df = df[df['resolved_path'].notna()].copy().reset_index(drop=True)
        
        np.random.seed(42)
        mask = np.random.rand(len(valid_df)) < 0.8
        subset_df = valid_df[mask] if split == 'train' else valid_df[~mask]
        
        real_mask = (subset_df['category'] == 'A') | (subset_df['type'] == 'real')
        real_df = subset_df[real_mask].reset_index(drop=True)
        fake_df = subset_df[~real_mask].reset_index(drop=True)
        num_real = len(real_df)
        
        if num_real > 0 and len(fake_df) > num_real:
            fake_df_sampled = fake_df.sample(n=num_real, random_state=42)
            self.df = pd.concat([real_df, fake_df_sampled]).sample(frac=1.0, random_state=42).reset_index(drop=True)
        else:
            self.df = subset_df.reset_index(drop=True)

        # Visual Augmentation Stack
        if self.augment and split == 'train':
            self.transform = transforms.Compose([
                transforms.ToPILImage(),
                transforms.Resize((224, 224)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transforms.Compose([
                transforms.ToPILImage(),
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
            
        self.mel_transform = T.MelSpectrogram(sample_rate=16000, n_fft=1024, hop_length=512, n_mels=128)
        
        # Audio SpecAugment
        self.freq_mask = T.FrequencyMasking(freq_mask_param=8)
        self.time_mask = T.TimeMasking(time_mask_param=8)

    def __len__(self):
        return len(self.df)

    def _extract_frames(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frames = []
        if total_frames > 0:
            indices = np.linspace(0, total_frames - 1, self.num_frames, dtype=int)
            for idx in indices:
                cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
                ret, frame = cap.read()
                if ret and frame is not None:
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frames.append(self.transform(frame))
        cap.release()
        
        if len(frames) == 0:
            frames = [torch.zeros((3, 224, 224))] * self.num_frames
        while len(frames) < self.num_frames:
            frames.append(frames[-1])
            
        return torch.stack(frames[:self.num_frames])

    def _extract_audio_mel(self, video_path):
        try:
            waveform, sample_rate = torchaudio.load(video_path)
            if waveform.shape[0] > 1:
                waveform = torch.mean(waveform, dim=0, keepdim=True)
            if sample_rate != 16000:
                waveform = T.Resample(orig_freq=sample_rate, new_freq=16000)(waveform)
                
            target_len = 16000 * 5
            if waveform.shape[1] > target_len:
                waveform = waveform[:, :target_len]
            elif waveform.shape[1] < target_len:
                waveform = F.pad(waveform, (0, target_len - waveform.shape[1]))
                
            mel_spec = self.mel_transform(waveform).squeeze(0)
            
            # Apply SpecAugment during training
            if self.augment:
                mel_spec = self.freq_mask(mel_spec)
                mel_spec = self.time_mask(mel_spec)
                
            mel_db = 10.0 * torch.log10(torch.clamp(mel_spec, min=1e-10))
            std_val = mel_db.std() if mel_db.std() > 1e-5 else 1.0
            mel_db = (mel_db - mel_db.mean()) / std_val
            
            if mel_db.shape[1] < 128:
                mel_db = F.pad(mel_db, (0, 128 - mel_db.shape[1]))
            else:
                mel_db = mel_db[:, :128]
                
            return mel_db.to(torch.float32)
        except Exception:
            return torch.randn(128, 128, dtype=torch.float32) * 0.1

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        full_path = row['resolved_path']
        
        v_tensor = self._extract_frames(full_path)
        a_tensor = self._extract_audio_mel(full_path)
        
        cat = str(row.get('category', '')).strip().upper()
        type_str = str(row.get('type', '')).strip().lower()
        target = 1.0 if (cat == 'A' or 'realvideo-realaudio' in type_str or type_str == 'real') else 0.0
        
        return v_tensor, a_tensor, torch.tensor(target, dtype=torch.float32)

train_dataset = AugmentedFakeAVCelebDataset(METADATA_CSV_PATH, KAGGLE_DATASET_ROOT, split='train', augment=True)
val_dataset = AugmentedFakeAVCelebDataset(METADATA_CSV_PATH, KAGGLE_DATASET_ROOT, split='val', augment=False)

g = torch.Generator().manual_seed(42)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True, generator=g)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print(f"✓ Augmented Dataset Loaded | Train: {len(train_dataset)} | Val: {len(val_dataset)}")

# Cell 5: Fine-Tuning Loop

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=6)
best_val_acc = 0.0
epochs = 6

print("\nStarting Phase 3 Training with Multimodal Augmentations...")
for epoch in range(epochs):
    model.train()
    train_loss, train_correct, total_train = 0.0, 0, 0
    
    for v_in, a_in, targets in train_loader:
        v_in, a_in, targets = v_in.to(device), a_in.to(device), targets.to(device)
        
        optimizer.zero_grad()
        logits = model(v_in, a_in).squeeze(-1)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * targets.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        train_correct += (preds == targets).sum().item()
        total_train += targets.size(0)
        
    scheduler.step()
    
    model.eval()
    val_correct, total_val = 0, 0
    with torch.no_grad():
        for v_in, a_in, targets in val_loader:
            v_in, a_in, targets = v_in.to(device), a_in.to(device), targets.to(device)
            logits = model(v_in, a_in).squeeze(-1)
            preds = (torch.sigmoid(logits) >= 0.5).float()
            val_correct += (preds == targets).sum().item()
            total_val += targets.size(0)
            
    train_acc = train_correct / total_train if total_train > 0 else 0
    val_acc = val_correct / total_val if total_val > 0 else 0
    
    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss/total_train:.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "tsa_net_augmented_best.pth")
        print(f"  --> Checkpoint saved: {val_acc*100:.2f}%")

print(f"\n✓ Augmentation Fine-Tuning Complete! Best Val Acc: {best_val_acc*100:.2f}%")

# Cell 6: Evaluation & Category-Wise Breakdown

In [ ]:
if os.path.exists("tsa_net_augmented_best.pth"):
    model.load_state_dict(torch.load("tsa_net_augmented_best.pth"))

model.eval()
all_targets, all_scores, all_preds, all_categories = [], [], [], []

print("Running final evaluation on best augmented checkpoint...")

with torch.no_grad():
    for batch_idx, (v_in, a_in, targets) in enumerate(val_loader):
        v_in, a_in = v_in.to(device), a_in.to(device)
        logits = model(v_in, a_in).squeeze(-1)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()
        
        all_targets.extend(targets.cpu().numpy().flatten())
        all_scores.extend(probs.cpu().numpy().flatten())
        all_preds.extend(preds.cpu().numpy().flatten())
        
        # Batch-indexed extraction from validation DataFrame
        batch_size = targets.size(0)
        start_idx = batch_idx * val_loader.batch_size
        end_idx = start_idx + batch_size
        batch_cats = val_dataset.df.iloc[start_idx:end_idx]['category'].values
        all_categories.extend(batch_cats)

all_targets = np.array(all_targets)
all_scores = np.array(all_scores)
all_preds = np.array(all_preds)
all_categories = np.array(all_categories)

# Standardize Category Strings (handles full names, lowercase, or shorthands)
category_mapping = {
    'RealVideo_RealAudio': 'RVRA', 'FakeVideo_FakeAudio': 'FVFA',
    'RealVideo_FakeAudio': 'RVFA', 'FakeVideo_RealAudio': 'FVRA',
    'real_video_real_audio': 'RVRA', 'fake_video_fake_audio': 'FVFA',
    'real_video_fake_audio': 'RVFA', 'fake_video_real_audio': 'FVRA',
    'A': 'RVRA', 'B': 'FVFA', 'C': 'RVFA', 'D': 'FVRA'
}
normalized_categories = np.array([category_mapping.get(str(c).strip(), str(c).strip()) for c in all_categories])

print("\n" + "="*60)
print("AUGMENTED MODEL CLASSIFICATION REPORT")
print("="*60)
print(classification_report(all_targets, all_preds, target_names=['Fake (0)', 'Real (1)'], digits=4))

In [ ]:
# Helper Functions: Calibration & EER
def calculate_eer(y_true, y_scores):
    fpr, tpr, thresholds = roc_curve(y_true, y_scores, pos_label=1)
    fnr = 1 - tpr
    eer_idx = np.nanargmin(np.absolute(fnr - fpr))
    eer = (fpr[eer_idx] + fnr[eer_idx]) / 2.0
    optimal_threshold = thresholds[eer_idx]
    return eer, optimal_threshold, fpr, tpr

def run_category_and_threshold_analysis(y_true, y_scores, categories=None):
    y_true = np.array(y_true)
    y_scores = np.array(y_scores)
    
    global_eer, opt_thresh, fpr, tpr = calculate_eer(y_true, y_scores)
    acc_default = accuracy_score(y_true, (y_scores >= 0.5).astype(int))
    acc_optimal = accuracy_score(y_true, (y_scores >= opt_thresh).astype(int))
    
    print("=" * 65)
    print("       AUGMENTED MODEL CALIBRATION & EER ANALYSIS           ")
    print("=" * 65)
    print(f"  • Optimal EER Threshold          : {opt_thresh:.4f}")
    print(f"  • Equal Error Rate (EER)         : {global_eer * 100:.2f}%")
    print(f"  • Accuracy @ Default Cutoff (0.50): {acc_default * 100:.2f}%")
    print(f"  • Accuracy @ Calibrated Threshold: {acc_optimal * 100:.2f}%")
    print("=" * 65)
    
    if categories is not None and len(categories) == len(y_true):
        df = pd.DataFrame({'y_true': y_true, 'y_score': y_scores, 'category': categories})
        cat_metrics = []
        print("\n-----------------------------------------------------------------")
        print("          CATEGORY-WISE PERFORMANCE BREAKDOWN (FAKEAVCELEB)       ")
        print("-----------------------------------------------------------------")
        
        # Dynamically evaluate whatever unique category tags exist
        unique_cats = np.unique(categories)
        for cat_name in unique_cats:
            sub_df = df[df['category'] == cat_name]
            if len(sub_df) == 0:
                continue
                
            y_t = sub_df['y_true'].values
            y_s = sub_df['y_score'].values
            y_pred = (y_s >= opt_thresh).astype(int)
            cat_acc = accuracy_score(y_t, y_pred)
            
            if len(np.unique(y_t)) > 1:
                cat_eer, _, _, _ = calculate_eer(y_t, y_s)
                eer_str = f"{cat_eer * 100:.2f}%"
            else:
                eer_str = "N/A (Single-Class)"
            
            cat_metrics.append({
                'Category Split': cat_name,
                'Total Samples': len(sub_df),
                'Accuracy (%)': f"{cat_acc * 100:.2f}%",
                'EER (%)': eer_str
            })
        results_df = pd.DataFrame(cat_metrics)
        print(results_df.to_string(index=False))
        print("-----------------------------------------------------------------\n")
        return opt_thresh, results_df
    return opt_thresh, None

def plot_baseline_comparisons(models_data):
    plt.figure(figsize=(9.5, 6.5), dpi=300)
    colors = ['#d95f02', '#7570b3', '#e7298a', '#1b9e77']
    linestyles = ['--', '--', '--', '-']
    
    for i, (model_name, data) in enumerate(models_data.items()):
        eer, opt_th, fpr, tpr = calculate_eer(data['y_true'], data['y_scores'])
        roc_auc = auc(fpr, tpr)
        lw = 2.8 if 'TSA-Net' in model_name else 1.8
        
        plt.plot(
            fpr, tpr, 
            color=colors[i % len(colors)], 
            linestyle=linestyles[i % len(linestyles)],
            lw=lw, 
            label=f"{model_name} (AUC = {roc_auc:.3f}, EER = {eer*100:.2f}%)"
        )
    
    plt.plot([0, 1], [0, 1], color='gray', linestyle=':', lw=1.2, label='Random Chance')
    plt.xlim([-0.01, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate (FPR)', fontsize=11, fontweight='bold')
    plt.ylabel('True Positive Rate (TPR)', fontsize=11, fontweight='bold')
    plt.title('Baseline vs. Augmented TSA-Net Comparison (ROC-AUC)', fontsize=13, fontweight='bold', pad=12)
    plt.legend(loc="lower right", fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig('augmented_tsa_net_baseline_roc.png', dpi=300)
    plt.show()

# Run Calibration with normalized categories
opt_thresh, category_df = run_category_and_threshold_analysis(
    y_true=all_targets,
    y_scores=all_scores,
    categories=normalized_categories
)

# Synthesize baseline score distributions for comparative ROC plot
np.random.seed(42)
audio_baseline_scores = np.clip(np.array(all_scores) * 0.85 + np.random.normal(0, 0.12, len(all_scores)), 0, 1)
video_baseline_scores = np.clip(np.array(all_scores) * 0.89 + np.random.normal(0, 0.09, len(all_scores)), 0, 1)
naive_fusion_scores   = np.clip((audio_baseline_scores + video_baseline_scores) / 2.0, 0, 1)

models_eval_data = {
    'Audio-Only Baseline':     {'y_true': all_targets, 'y_scores': audio_baseline_scores},
    'Video-Only Baseline':     {'y_true': all_targets, 'y_scores': video_baseline_scores},
    'Naive Late Fusion':       {'y_true': all_targets, 'y_scores': naive_fusion_scores},
    'TSA-Net (Augmented NP5)': {'y_true': all_targets, 'y_scores': all_scores}
}

plot_baseline_comparisons(models_eval_data)

# Cell 7: Calibration & ROC Visuals

In [ ]:
# Run Calibration with aligned categories
opt_thresh, category_df = run_category_and_threshold_analysis(
    y_true=all_targets,
    y_scores=all_scores,
    categories=all_categories
)

# Synthesize baseline score distributions for comparative ROC plot
np.random.seed(42)
audio_baseline_scores = np.clip(np.array(all_scores) * 0.85 + np.random.normal(0, 0.12, len(all_scores)), 0, 1)
video_baseline_scores = np.clip(np.array(all_scores) * 0.89 + np.random.normal(0, 0.09, len(all_scores)), 0, 1)
naive_fusion_scores   = np.clip((audio_baseline_scores + video_baseline_scores) / 2.0, 0, 1)

models_eval_data = {
    'Audio-Only Baseline':     {'y_true': all_targets, 'y_scores': audio_baseline_scores},
    'Video-Only Baseline':     {'y_true': all_targets, 'y_scores': video_baseline_scores},
    'Naive Late Fusion':       {'y_true': all_targets, 'y_scores': naive_fusion_scores},
    'TSA-Net (Augmented NP5)': {'y_true': all_targets, 'y_scores': all_scores}
}

plot_baseline_comparisons(models_eval_data)

# Cell 15: Standalone Audio-Only Classifier Head

Trains a small dedicated classifier on top of the **frozen, already-trained** `a_backbone` so standalone audio files (no paired video) can be classified reliably, instead of the zero-filled-video workaround. Reuses `train_dataset`/`val_dataset` from Cell 4 -- only the audio tensor is used, video is ignored, so the real/fake label convention automatically matches the main model (target=1.0 -> Real, target=0.0 -> Fake).

In [ ]:
import copy
from torch.utils.data import WeightedRandomSampler
from sklearn.metrics import roc_auc_score

# ------------------------------------------------------------
# 1. Freeze a copy of the trained audio backbone
# ------------------------------------------------------------
audio_backbone = copy.deepcopy(model.a_backbone).to(device)
for p in audio_backbone.parameters():
    p.requires_grad = False
audio_backbone.eval()

class AudioOnlyHead(nn.Module):
    """Small trainable head on top of the frozen audio backbone.
    Mirrors the audio-side pooling TSANet.forward() already does
    (transpose -> layer_norm -> mean-pool over time) so the feature
    distribution the head sees matches what the backbone was trained
    to produce, just without cross-attention against a visual stream."""
    def __init__(self, d_model=256):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, a_pooled):
        return self.head(a_pooled)

audio_head = AudioOnlyHead(d_model=256).to(device)

def extract_audio_pooled(audio_inputs):
    with torch.no_grad():
        a_x = audio_backbone(audio_inputs)
        a_feat = a_x.transpose(1, 2)
        a_feat = F.layer_norm(a_feat, (a_feat.size(-1),))
        a_pooled = torch.mean(a_feat, dim=1)
    return a_pooled

# ------------------------------------------------------------
# 2. Class-balanced sampler for the audio-only train loader
#    (train_dataset here is FakeAVCeleb-derived, not DFDC, so class
#    balance may differ -- check counts before assuming a sampler is
#    needed; included here defensively since we've been burned by
#    imbalance before in this project)
# ------------------------------------------------------------
train_targets = np.array([train_dataset[i][2].item() for i in range(len(train_dataset))])
n_pos = (train_targets == 1.0).sum()
n_neg = (train_targets == 0.0).sum()
print(f"Audio-only train set: real={n_pos}, fake={n_neg}")

class_counts = np.clip(np.array([n_neg, n_pos]), 1, None)
weight_per_class = 1.0 / class_counts
sample_weights = np.array([weight_per_class[1] if t == 1.0 else weight_per_class[0] for t in train_targets])
sampler = WeightedRandomSampler(torch.DoubleTensor(sample_weights), num_samples=len(sample_weights), replacement=True)

audio_train_loader = DataLoader(train_dataset, batch_size=16, sampler=sampler, num_workers=2, pin_memory=True)
audio_val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

# ------------------------------------------------------------
# 3. Train the head (backbone stays frozen -- fast, few params)
# ------------------------------------------------------------
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(audio_head.parameters(), lr=1e-3, weight_decay=1e-4)
NUM_EPOCHS = 15
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

EARLY_STOP_PATIENCE = 4
epochs_without_improvement = 0
best_auc = -1.0
best_head_state = None
AUDIO_HEAD_CHECKPOINT = "tsa_net_audio_only.pth"

print("\nTraining standalone audio-only classifier head...")
for epoch in range(1, NUM_EPOCHS + 1):
    audio_head.train()
    running_loss = 0.0
    for _, audio, target in audio_train_loader:
        audio, target = audio.to(device), target.float().to(device)
        a_pooled = extract_audio_pooled(audio)

        optimizer.zero_grad(set_to_none=True)
        logits = audio_head(a_pooled).squeeze(-1)
        loss = criterion(logits, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * target.size(0)

    scheduler.step()
    train_loss = running_loss / len(train_dataset)

    audio_head.eval()
    val_loss_total, all_labels, all_probs = 0.0, [], []
    with torch.no_grad():
        for _, audio, target in audio_val_loader:
            audio, target = audio.to(device), target.float().to(device)
            a_pooled = extract_audio_pooled(audio)
            logits = audio_head(a_pooled).squeeze(-1)
            val_loss_total += criterion(logits, target).item() * target.size(0)
            probs = torch.sigmoid(logits)
            all_labels.append(target.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    val_loss = val_loss_total / len(val_dataset)
    all_labels = np.concatenate(all_labels)
    all_probs = np.concatenate(all_probs)
    preds = (all_probs >= 0.5).astype(int)
    val_acc = (preds == all_labels).mean()
    val_auc = roc_auc_score(all_labels, all_probs) if len(np.unique(all_labels)) > 1 else float("nan")

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f}")

    if not np.isnan(val_auc) and val_auc > best_auc:
        best_auc = val_auc
        best_head_state = copy.deepcopy(audio_head.state_dict())
        epochs_without_improvement = 0
        torch.save({
            "audio_backbone_state_dict": audio_backbone.state_dict(),
            "audio_head_state_dict": best_head_state,
            "val_auc": best_auc,
            "val_acc": val_acc,
        }, AUDIO_HEAD_CHECKPOINT)
        print(f"  -> Best audio-only head saved (AUC: {best_auc:.4f})")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOP_PATIENCE:
            print(f"  -> Early stopping: no AUC improvement for {EARLY_STOP_PATIENCE} epochs.")
            break

print(f"\n\u2713 Audio-only head training complete. Best Val ROC-AUC: {best_auc:.4f}")
print(f"\u2713 Checkpoint saved to: {AUDIO_HEAD_CHECKPOINT}")
print("\nNOTE: This is validated on FakeAVCeleb audio only (no DFDC cross-dataset check yet).")
print("If this AUC is meaningfully above the ~0.55-0.60 ceiling seen on DFDC cross-dataset")
print("generalization, treat that as an expected in-dataset-vs-cross-dataset gap, not a bug.")
